In [ ]:
import os

# Reliable hosted-Colab detection:
# - /content exists only in Google's managed VMs (not on your local machine)
# - TBE_CREDS_ADDR is set by Google's credential server, present only in hosted runtimes
#   (this is the same check google.colab.drive.mount uses internally)
# COLAB_RELEASE_TAG is NOT reliable — Colab's frontend injects it into all connected
# kernels, including local runtimes, making it useless for this distinction.
IN_HOSTED_COLAB = os.path.exists('/content') and bool(os.environ.get('TBE_CREDS_ADDR'))

if IN_HOSTED_COLAB:
    !git clone https://github.com/jalengg/groundwork.git /content/groundwork
    %cd /content/groundwork
    !pip install -r requirements.txt -q
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs('/content/drive/MyDrive/groundwork/data', exist_ok=True)
    os.makedirs('/content/drive/MyDrive/groundwork/checkpoints', exist_ok=True)
    !ln -sf /content/drive/MyDrive/groundwork/data data
    !ln -sf /content/drive/MyDrive/groundwork/checkpoints checkpoints
else:
    # Local runtime — already running from the groundwork directory
    %cd /mnt/c/Users/jalen/groundwork
    print(f"Working directory: {os.getcwd()}")

In [ ]:
!python model/train_diffusion.py \
    --vae checkpoints/vae/vae_epoch_050.pth \
    --data data/ \
    --output checkpoints/diffusion/ \
    --epochs 200 \
    --batch 4

In [ ]:
import torch, numpy as np, os
from model.vae import RoadVAE
from model.unet import DiffusionUNet
from model.diffusion import DDPM
from model.vlm_eval import score_samples

EPOCH = 100  # change this
device = torch.device('cuda')

vae = RoadVAE().to(device)
vae.load_state_dict(torch.load('checkpoints/vae/vae_epoch_050.pth')['model'])
vae.eval()

net = DiffusionUNet().to(device)
net.load_state_dict(torch.load(f'checkpoints/diffusion/diffusion_epoch_{EPOCH:03d}.pth')['model'])
net.eval()

ddpm = DDPM()
# Generate 10 unconditional samples (cond=zeros)
samples = []
for _ in range(10):
    with torch.no_grad():
        cond = torch.zeros(1, 4, 512, 512, device=device)
        latent = ddpm.sample_ddim(net, cond, n_steps=50)
        road = vae.decode(latent)
        road_argmax = road[0].argmax(0).cpu().numpy()
        one_hot = (np.arange(5)[:, None, None] == road_argmax[None]).astype(np.float32)
        samples.append(one_hot)

# VLM scoring
os.environ['ANTHROPIC_API_KEY'] = 'YOUR_KEY_HERE'
scores = score_samples(samples)
avg = sum(s['score'] for s in scores) / len(scores)
print(f"Epoch {EPOCH} — Avg VLM realism score: {avg:.1f}/10")
for i, s in enumerate(scores):
    print(f"  Sample {i+1}: {s['score']}/10 — {s['issues']}")

In [ ]:
# Vectorize samples and compute CI / TC
from vectorize.postprocess import vectorize_road_layout
from model.eval_metrics import compute_connectivity_index, compute_transport_convenience

ci_scores, tc_scores = [], []
for sample in samples:
    G = vectorize_road_layout(sample)
    if G and G.number_of_nodes() > 5:
        ci_scores.append(compute_connectivity_index(G))
        tc_scores.append(compute_transport_convenience(G))

print(f"CI: {np.mean(ci_scores):.3f} (target > 1.8)")
print(f"TC: {np.mean(tc_scores):.3f} (target > 0.6)")
# CaRoLS benchmark: CI=1.948, TC=0.668